In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Configuration & Constants ----------------------------------------------
# World definition
WIDTH = 1.0
HEIGHT = 1.0

# Physics constants
MAX_SPEED = 0.01
MIN_SPEED = 0.003
NEIGHBOR_RADIUS = 0.1
SEPARATION_RADIUS = 0.03
DT = 0.05  # Time step

# Rule weights
SEPARATION_WEIGHT = 0.15
ALIGNMENT_WEIGHT = 0.05
COHESION_WEIGHT = 0.01

# --- Core Simulation Logic --------------------------------------------------

def init_boids(n_boids, width, height):
    """Initialize boids with random positions and velocities."""
    positions = np.random.rand(n_boids, 2) * np.array([width, height])

    # Random velocities: random angle, speed between MIN and MAX
    angles = np.random.rand(n_boids) * 2 * np.pi
    speeds = (np.random.rand(n_boids) * (MAX_SPEED - MIN_SPEED)) + MIN_SPEED
    velocities = np.column_stack((np.cos(angles), np.sin(angles))) * speeds[:, None]

    return positions, velocities

def update_boids(positions, velocities, params):
    """Perform one update step using vectorized NumPy operations."""
    width = params["width"]
    height = params["height"]
    n_boids = positions.shape[0]

    # 1. Calculate pairwise difference vectors (Torus topology)
    # diff[i, j] is the vector from boid i to boid j
    diff = positions[None, :, :] - positions[:, None, :]

    # Apply torus wrapping (shortest path)
    diff[:, :, 0] = (diff[:, :, 0] + width / 2) % width - width / 2
    diff[:, :, 1] = (diff[:, :, 1] + height / 2) % height - height / 2

    # Calculate squared distances
    dist_sq = np.sum(diff**2, axis=2)
    np.fill_diagonal(dist_sq, np.inf) # Ignore self-interaction

    # 2. Identify neighbors
    neighbor_mask = dist_sq < params["neighbor_radius"]**2
    separation_mask = dist_sq < params["separation_radius"]**2

    neighbor_counts = np.sum(neighbor_mask, axis=1, keepdims=True)
    # Avoid division by zero
    neighbor_counts_safe = np.where(neighbor_counts == 0, 1, neighbor_counts)

    # 3. Apply Boids Rules

    # Separation: Move away from close neighbors (weighted by 1/dist)
    sep_vectors = -diff * separation_mask[:, :, None]
    separation = np.sum(sep_vectors / (dist_sq[:, :, None] + 1e-8), axis=1)

    # Alignment: Steer towards average velocity of neighbors
    avg_vel = np.sum(velocities[None, :, :] * neighbor_mask[:, :, None], axis=1) / neighbor_counts_safe
    alignment = avg_vel - velocities

    # Cohesion: Steer towards center of mass (average relative position)
    avg_rel_pos = np.sum(diff * neighbor_mask[:, :, None], axis=1) / neighbor_counts_safe
    cohesion = avg_rel_pos

    # Zero out rules for boids with no neighbors
    no_neighbors = (neighbor_counts[:, 0] == 0)
    alignment[no_neighbors] = 0
    cohesion[no_neighbors] = 0

    # 4. Apply forces
    acceleration = (
        separation * params["sep_weight"] +
        alignment * params["align_weight"] +
        cohesion * params["coh_weight"]
    )

    velocities += acceleration * params["dt"]

    # 5. Limit Speeds
    speeds = np.linalg.norm(velocities, axis=1, keepdims=True)
    speeds = np.maximum(speeds, 1e-8)
    scale_factor = np.where(speeds > params["max_speed"], params["max_speed"] / speeds, 1.0)
    velocities *= scale_factor

    # 6. Update Positions and Wrap
    positions += velocities * params["dt"]
    positions[:, 0] %= width
    positions[:, 1] %= height

    return positions, velocities

# --- Colab Execution & Rendering --------------------------------------------

def run_boids_colab(n_boids=100, frames=200, fps=30):
    """
    Runs the simulation and returns an interactive HTML video.
    """
    # Parameter dictionary
    params = {
        "width": WIDTH, "height": HEIGHT, "dt": DT,
        "max_speed": MAX_SPEED, "neighbor_radius": NEIGHBOR_RADIUS,
        "separation_radius": SEPARATION_RADIUS,
        "sep_weight": SEPARATION_WEIGHT, "align_weight": ALIGNMENT_WEIGHT,
        "coh_weight": COHESION_WEIGHT,
    }

    print("-" * 40)
    print(f"Initializing Boids Simulation (N={n_boids})...")

    # Initialize State
    positions, velocities = init_boids(n_boids, WIDTH, HEIGHT)

    # Setup Plot
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(0, WIDTH)
    ax.set_ylim(0, HEIGHT)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"Boids (N={n_boids})")

    # Quiver plot (arrows)
    quiver = ax.quiver(
        positions[:, 0], positions[:, 1],
        velocities[:, 0], velocities[:, 1],
        angles='xy', scale_units='xy', scale=1.0/DT,
        width=0.005, headwidth=3, headlength=4, color='teal'
    )

    def animate(frame):
        # Use nonlocal to modify state variables in parent scope
        nonlocal positions, velocities
        positions, velocities = update_boids(positions, velocities, params)

        # Update graphics
        quiver.set_offsets(positions)
        quiver.set_UVC(velocities[:, 0], velocities[:, 1])
        return quiver,

    print(f"Rendering {frames} frames. Please wait...")

    anim = FuncAnimation(
        fig, animate,
        frames=frames, interval=1000/fps, blit=True
    )

    # Close the static plot to prevent double-display
    plt.close(fig)

    # Return the interactive HTML object
    return HTML(anim.to_jshtml())

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Configuration & Constants ----------------------------------------------
# World definition
WIDTH = 1.0
HEIGHT = 1.0

# Physics constants
MAX_SPEED = 0.01
MIN_SPEED = 0.003
NEIGHBOR_RADIUS = 0.1
SEPARATION_RADIUS = 0.03
DT = 0.05  # Time step

# Rule weights
SEPARATION_WEIGHT = 0.15
ALIGNMENT_WEIGHT = 0.05
COHESION_WEIGHT = 0.01

# --- Core Simulation Logic --------------------------------------------------

def init_boids(n_boids, width, height):
    """Initialize boids with random positions and velocities."""
    positions = np.random.rand(n_boids, 2) * np.array([width, height])

    # Random velocities: random angle, speed between MIN and MAX
    angles = np.random.rand(n_boids) * 2 * np.pi
    speeds = (np.random.rand(n_boids) * (MAX_SPEED - MIN_SPEED)) + MIN_SPEED
    velocities = np.column_stack((np.cos(angles), np.sin(angles))) * speeds[:, None]

    return positions, velocities

def update_boids(positions, velocities, params):
    """Perform one update step using vectorized NumPy operations."""
    width = params["width"]
    height = params["height"]
    n_boids = positions.shape[0]

    # 1. Calculate pairwise difference vectors (Torus topology)
    # diff[i, j] is the vector from boid i to boid j
    diff = positions[None, :, :] - positions[:, None, :]

    # Apply torus wrapping (shortest path)
    diff[:, :, 0] = (diff[:, :, 0] + width / 2) % width - width / 2
    diff[:, :, 1] = (diff[:, :, 1] + height / 2) % height - height / 2

    # Calculate squared distances
    dist_sq = np.sum(diff**2, axis=2)
    np.fill_diagonal(dist_sq, np.inf) # Ignore self-interaction

    # 2. Identify neighbors
    neighbor_mask = dist_sq < params["neighbor_radius"]**2
    separation_mask = dist_sq < params["separation_radius"]**2

    neighbor_counts = np.sum(neighbor_mask, axis=1, keepdims=True)
    # Avoid division by zero
    neighbor_counts_safe = np.where(neighbor_counts == 0, 1, neighbor_counts)

    # 3. Apply Boids Rules

    # Separation: Move away from close neighbors (weighted by 1/dist)
    sep_vectors = -diff * separation_mask[:, :, None]
    separation = np.sum(sep_vectors / (dist_sq[:, :, None] + 1e-8), axis=1)

    # Alignment: Steer towards average velocity of neighbors
    avg_vel = np.sum(velocities[None, :, :] * neighbor_mask[:, :, None], axis=1) / neighbor_counts_safe
    alignment = avg_vel - velocities

    # Cohesion: Steer towards center of mass (average relative position)
    avg_rel_pos = np.sum(diff * neighbor_mask[:, :, None], axis=1) / neighbor_counts_safe
    cohesion = avg_rel_pos

    # Zero out rules for boids with no neighbors
    no_neighbors = (neighbor_counts[:, 0] == 0)
    alignment[no_neighbors] = 0
    cohesion[no_neighbors] = 0

    # 4. Apply forces
    acceleration = (
        separation * params["sep_weight"] +
        alignment * params["align_weight"] +
        cohesion * params["coh_weight"]
    )

    velocities += acceleration * params["dt"]

    # 5. Limit Speeds
    speeds = np.linalg.norm(velocities, axis=1, keepdims=True)
    speeds = np.maximum(speeds, 1e-8)
    scale_factor = np.where(speeds > params["max_speed"], params["max_speed"] / speeds, 1.0)
    velocities *= scale_factor

    # 6. Update Positions and Wrap
    positions += velocities * params["dt"]
    positions[:, 0] %= width
    positions[:, 1] %= height

    return positions, velocities

# --- Colab Execution & Rendering --------------------------------------------

def run_boids_colab(n_boids=100, frames=200, fps=30):
    """
    Runs the simulation and returns an interactive HTML video.
    """
    # Parameter dictionary
    params = {
        "width": WIDTH, "height": HEIGHT, "dt": DT,
        "max_speed": MAX_SPEED, "neighbor_radius": NEIGHBOR_RADIUS,
        "separation_radius": SEPARATION_RADIUS,
        "sep_weight": SEPARATION_WEIGHT, "align_weight": ALIGNMENT_WEIGHT,
        "coh_weight": COHESION_WEIGHT,
    }

    print("-" * 40)
    print(f"Initializing Boids Simulation (N={n_boids})...")

    # Initialize State
    positions, velocities = init_boids(n_boids, WIDTH, HEIGHT)

    # Setup Plot
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(0, WIDTH)
    ax.set_ylim(0, HEIGHT)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"Boids (N={n_boids})")

    # Quiver plot (arrows)
    quiver = ax.quiver(
        positions[:, 0], positions[:, 1],
        velocities[:, 0], velocities[:, 1],
        angles='xy', scale_units='xy', scale=1.0/DT,
        width=0.005, headwidth=3, headlength=4, color='teal'
    )

    def animate(frame):
        # Use nonlocal to modify state variables in parent scope
        nonlocal positions, velocities
        positions, velocities = update_boids(positions, velocities, params)

        # Update graphics
        quiver.set_offsets(positions)
        quiver.set_UVC(velocities[:, 0], velocities[:, 1])
        return quiver,

    print(f"Rendering {frames} frames. Please wait...")

    anim = FuncAnimation(
        fig, animate,
        frames=frames, interval=1000/fps, blit=True
    )

    # Close the static plot to prevent double-display
    plt.close(fig)

    # Return the interactive HTML object
    return HTML(anim.to_jshtml())

# Call the simulation function to run it when the cell is executed
run_boids_colab()

Output hidden; open in https://colab.research.google.com to view.